# Task 2: Bayesian Change Point Detection — Brent Oil Prices

**Project:** Birhan Energies — Change Point Analysis  
**Objective:** Apply PyMC Bayesian change point models to detect structural breaks in Brent oil log returns and associate them with curated market events.

## Model Specification

Single mean-shift change point model on daily log returns:

- **τ ~ DiscreteUniform(0, N−1)** — switch point index
- **μ₁, μ₂ ~ Normal(0, 1)** — segment means before/after τ
- **σ ~ HalfNormal(1)** — observation noise
- **μₜ = switch(t < τ, μ₁, μ₂)**
- **rₜ ~ Normal(μₜ, σ)** — likelihood on log returns

We analyze five event-aligned sub-periods where convergence is reliable, rather than the full 35-year series (which mixes many overlapping regime shifts).

In [ ]:
import json
import sys
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_brent_prices, load_events, compute_log_returns
from src.change_point_model import (
    build_change_point_model,
    fit_change_point_model,
    associate_change_point_with_events,
    price_level_impact,
)

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Build the PyMC Change Point Model

Following the challenge specification: discrete uniform prior on τ, Normal priors on segment means, and `pm.math.switch` for the piecewise mean.

In [ ]:
prices = load_brent_prices()
events = load_events()
returns = compute_log_returns(prices)

# Example: COVID 2020 window
covid_returns = returns["2020-01-01":"2020-12-31"]
data = covid_returns.values.astype(float)

model = build_change_point_model(data)
pm.model_to_graphviz(model)

## 2. Run MCMC and Check Convergence

In [ ]:
result = fit_change_point_model(covid_returns, draws=2000, tune=1500, chains=2)

print("=== Convergence Diagnostics ===")
display(result.summary)

print(f"\nChange point (median): {result.tau_summary['tau_median_date']}")
print(f"94% HDI: {result.tau_summary['tau_hdi_lower_date']} – {result.tau_summary['tau_hdi_upper_date']}")
print(f"P(μ₂ > μ₁): {result.impact_summary['prob_mu2_greater_mu1']:.3f}")

## 3. Posterior Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, param, color in zip(axes, ["tau", "mu_1", "mu_2"], ["#2ecc71", "#3498db", "#e74c3c"]):
    samples = result.trace.posterior[param].values.flatten()
    if param == "tau":
        samples = result.dates[samples.astype(int)]
    ax.hist(samples, bins=40, color=color, edgecolor="white")
    ax.set_title(f"Posterior: {param}")
    if param == "tau":
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.tick_params(axis="x", rotation=45)
plt.suptitle("COVID 2020 — Change Point Posteriors", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. All Period Analyses — Results Summary

Pre-computed results from `scripts/run_change_point_analysis.py` (also exported to `outputs/change_point_results.json` for the dashboard).

In [ ]:
with open(PROJECT_ROOT / "outputs" / "change_point_results.json") as f:
    all_results = json.load(f)

summary_rows = []
for r in all_results:
    summary_rows.append({
        "Period": r["name"],
        "Change Point": r["tau_summary"]["tau_median_date"],
        "94% HDI": f"{r['tau_summary']['tau_hdi_lower_date']} – {r['tau_summary']['tau_hdi_upper_date']}",
        "Pre Mean Price ($)": r["price_impact"]["pre_period_mean_price"],
        "Post Mean Price ($)": r["price_impact"]["post_period_mean_price"],
        "Price Change (%)": r["price_impact"]["price_pct_change"],
        "P(μ₂ > μ₁)": round(r["impact_summary"]["prob_mu2_greater_mu1"], 3),
        "R-hat (τ)": float(r["convergence"]["r_hat"]["tau"]),
    })

results_df = pd.DataFrame(summary_rows)
display(results_df)

## 5. Event Association

In [ ]:
associations = pd.read_csv(PROJECT_ROOT / "outputs" / "event_associations.csv")
closest = associations[associations["within_window"] == True].sort_values("days_from_change_point")
display(closest)

print("\nEvents within 60-day window of detected change points:")
for _, row in closest.iterrows():
    print(f"  [{row['analysis']}] {row['event_name']} — {row['days_from_change_point']} days from τ")

## 6. Interpretation — Quantified Impact Statements

### COVID-19 and OPEC Response (2020)
Following the OPEC+ historic production cut announcement on **12 April 2020**, the model detects a change point on **21 April 2020** (9 days later, within the 94% HDI). Before the break, daily log returns averaged **μ₁ = −0.020** (sustained price decline); after the break, **μ₂ = +0.006** (stabilization/recovery). The posterior probability that μ₂ > μ₁ is **93.8%**. Average Brent price shifted from **$45.11 to $40.42** (−10.4%) across the full 2020 window, reflecting the demand collapse followed by partial recovery after coordinated OPEC+ cuts.

### Global Financial Crisis (2008–2009)
The model identifies a change point on **10 December 2008**, within the 94% HDI that spans the Lehman Brothers collapse (15 September 2008). Average Brent price fell from **$100.36 to $50.40**, a **−49.8%** decline — the largest price-level shift in our analysis.

### Ukraine War (2022)
A change point on **21 April 2022** is temporally consistent with both the **Russia invasion (24 Feb, 56 days)** and the **OPEC+ output decision (31 Mar, 21 days)**. Before τ, returns were strongly positive (μ₁ = +0.006, price spike phase); after τ, returns turned negative (μ₂ = −0.002, P(μ₂ > μ₁) = 11.5%), indicating the initial shock premium began to fade.

### Gulf War (1990–1991)
Change point detected on **5 October 1990**, 64 days after Iraq's invasion of Kuwait (2 August 1990), within the 94% HDI. This marks the transition from the initial price spike (μ₁ = +0.007) to post-spike stabilization (μ₂ = −0.005).

### OPEC Price War (2014–2016)
Change point on **19 January 2016** marks the transition near the bottom of the price war ($66 → $41 average, −38.5%). This precedes the September 2016 OPEC+ cut agreement, suggesting the model captured the end of the decline phase rather than the November 2014 OPEC decision itself.

---

> **Caution:** These are temporal associations, not causal proofs. See `docs/assumptions_and_limitations.md` for full discussion.